# ✂️ Chunk Scouting Reports for Embedding

Cohere’s `embed-english-v3` model (served through AWS Bedrock) has a maximum input length of **2048 characters**. To support this, we need to split long scouting reports into smaller, model-friendly chunks.

This notebook:
- Loads the full scouting report table created in the previous step
- Splits the `Combined_Scouting_Report` column into ~2048-character chunks
- Creates a new Delta table with one row per chunk, including:
  - Player metadata
  - Chunk index
  - Chunk text
  - Primary key (ID + season + chunk index)

These chunks will be used as the input to the Cohere embedding model during Delta Sync in the next step.


In [0]:
# Environment Variables
CATALOG = "alexander_booth"
SCHEMA = "cohere_demo"
TABLE = "fangraphs_mlb_scouting_reports"
TABLE_CHUNKS = "fangraphs_mlb_scouting_reports_chunked"

In [0]:
# Imports
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, StringType
from pyspark.sql import functions as F

In [0]:
# Load the source table containing full scouting reports
df = spark.table(f"{CATALOG}.{SCHEMA}.{TABLE}")

In [0]:
# Count the number of rows with text longer than 2048 characters
df.filter(F.length(F.col("Combined_Scouting_Report")) > 2048).count()

150

In [0]:
# Define a function to split long text into chunks of up to 2048 characters
def chunk_text(text, max_chars=2048):
    if text is None:
        return []  # Return an empty list for null input
    return [text[i:i + max_chars] for i in range(0, len(text), max_chars)]

# Register the function as a PySpark UDF that returns an array of strings
chunk_text_udf = udf(chunk_text, ArrayType(StringType()))

In [0]:
# Apply chunking and flatten the result so each chunk is its own row
df_chunked = (
    df.withColumn("content_chunks", chunk_text_udf(F.col("Combined_Scouting_Report")))  # Apply chunking to text column
      .select("ID", "playerName", "Season", F.posexplode("content_chunks").alias("chunk_index", "content_chunk")) # one row per index
      .withColumn("primary_key", F.concat_ws("_", F.col("ID"), F.col("Season"), F.col("chunk_index")))
)

# Save the chunked output to a new Delta table for use in embedding/vector search
df_chunked.write.mode("overwrite").saveAsTable(f"{CATALOG}.{SCHEMA}.{TABLE_CHUNKS}")

In [0]:
# Enable Change Data Feed (CDF) on the scouting reports chunks table using env vars
spark.sql(f"""
  ALTER TABLE `{CATALOG}`.`{SCHEMA}`.`{TABLE_CHUNKS}`
  SET TBLPROPERTIES (delta.enableChangeDataFeed = true)
""")


DataFrame[]

---

✅ **Next Step: Create the Vector Index with Delta Sync**

Now that the scouting reports are chunked into 2048-character segments, you can use the Databricks **Vector Search UI** to create a vector index backed by Delta Sync.

**To do this:**
1. Navigate to the **Vector Search** tab in the Databricks workspace
2. Click **"Create Vector Index"**
3. Select the chunked table (e.g., `fangraphs_mlb_scouting_reports_chunked`)
4. Choose the `content_chunk` column as the input text
5. Select your external **Cohere embedding endpoint** (e.g., `cohere_embed_english_v3`)
6. Set the primary key (e.g., `ID_Season_ChunkIndex`) and any metadata columns (e.g., player name, season)

Databricks will automatically embed and sync the content to power fast, semantic search.
